## Links

https://docs.langchain.com/oss/python/integrations/document_loaders

Documents and Document Loading

In [35]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [36]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./example-docs/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

107


In [37]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑  ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
F

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': './example-docs/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}


Text Splitting

In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

516


In [39]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:latest", base_url="http://192.168.2.202:11434")

In [40]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 4096

[-0.017552039, 0.019394705, -0.04626786, -0.05088339, -0.009332266, 0.04777892, 0.004180682, 0.007870216, -0.020839643, -0.0004072595]


Vector Store (In Memory)

In [41]:
from langchain_core.vectorstores import InMemoryVectorStore

memory_store = InMemoryVectorStore(embeddings)

In [42]:
ids = memory_store.add_documents(documents=all_splits)

In [43]:
results = await memory_store.asimilarity_search_with_score(
    "How many distribution centers does Nike have in the US?"
)
doc, score = results[0]
print(doc)
print(f"Score: {score}")

page_content='direct to consumer operations sell products through the following number of retail stores in the United States:
U.S. RETAIL STORES NUMBER
NIKE Brand factory stores 213 
NIKE Brand in-line stores (including employee-only stores) 74 
Converse stores (including factory stores) 82 
TOTAL 369 
In the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.
2023 FORM 10-K 2' metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': './example-docs/nke-10k-2023.pdf', 'total_pages': 107, 'page': 4, 'page_label': '5', 'start_index': 3125}
Score: 0.8338219794881716

Vector Store (ChromaDB)

In [44]:
from langchain_chroma import Chroma

db_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [45]:
ids = db_store.add_documents(documents=all_splits)

In [46]:
results = await db_store.asimilarity_search_with_score(
    "How many distribution centers does Nike have in the US?"
)

doc, score = results[0]
print(doc.page_content)
print(doc.metadata)
print(f"Score: {score}")

direct to consumer operations sell products through the following number of retail stores in the United States:
U.S. RETAIL STORES NUMBER
NIKE Brand factory stores 213 
NIKE Brand in-line stores (including employee-only stores) 74 
Converse stores (including factory stores) 82 
TOTAL 369 
In the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.
2023 FORM 10-K 2
{'page': 4, 'title': '0000320187-23-000039', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'creationdate': '2023-07-20T16:22:00-04:00', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'page_label': '5', 'total_pages': 107, 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'start_index': 3125, 'creator': 'EDGAR Filing HTML Converter', 'source': './example-docs/nke-10k-2023.pdf'}
Score: 0.3323562443256378
